# Technical Note P6: NumPy Advanced Array Operations

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{shape transformation}
+
\text{axis management}
+
\text{memory awareness}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. Reshape and index ordering

Reshape changes the index structure of an existing linear ordering. It does not rotate coordinates, interpolate a field, or change its physical meaning.

$$
12
\longrightarrow
3\times4
$$

In [2]:
import numpy as np

a = np.arange(12)
A = a.reshape(3, 4)

print("a =", a)
print("A =")
print(A)
print("a shape =", a.shape)
print("A shape =", A.shape)

a = [ 0  1  2  3  4  5  6  7  8  9 10 11]
A =
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
a shape = (12,)
A shape = (3, 4)


$$
N_1N_2\cdots N_k=
M_1M_2\cdots M_\ell
$$

## 2. Infer one dimension

A single minus-one dimension lets NumPy infer a compatible length. The total entry count must remain unchanged.

```python
A.reshape(3, -1)
```

In [3]:
a = np.arange(24)

A = a.reshape(3, -1)
B = a.reshape(2, 3, -1)

print("A shape =", A.shape)
print("B shape =", B.shape)

A shape = (3, 8)
B shape = (2, 3, 4)


## 3. Ravel and flatten

Ravel may share storage, whereas flatten copies. Verify aliasing when modifying a returned array.

```python
ravel()
flatten()
```

In [4]:
A = np.arange(12).reshape(3, 4)

r = A.ravel()
f = A.flatten()

print("shares memory A-r?", np.shares_memory(A, r))
print("shares memory A-f?", np.shares_memory(A, f))

shares memory A-r? True
shares memory A-f? False


$$
U_{ij}
\leftrightarrow
u_I
$$

## 4. C and Fortran ordering

C-order varies the last index fastest; Fortran-order varies the first fastest. Matrix assembly and field reconstruction must use the same order.

```python
A.ravel(order="C")
```

```python
A.ravel(order="F")
```

In [5]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
])

print("C order:", A.ravel(order="C"))
print("F order:", A.ravel(order="F"))

C order: [1 2 3 4 5 6]
F order: [1 4 2 5 3 6]


$$
\boxed{
\text{field indexing convention}
\leftrightarrow
\text{global vector ordering}
}
$$

## 5. Concatenate

Concatenate joins along an existing axis. Other axis lengths must agree.

In [6]:
a = np.array([1, 2, 3])
b = np.array([4, 5])

c = np.concatenate([a, b])

print(c)
print("shape =", c.shape)

[1 2 3 4 5]
shape = (5,)


In [7]:
A = np.ones((2, 3))
B = 2*np.ones((2, 3))

C0 = np.concatenate([A, B], axis=0)
C1 = np.concatenate([A, B], axis=1)

print("axis=0 shape =", C0.shape)
print("axis=1 shape =", C1.shape)

axis=0 shape = (4, 3)
axis=1 shape = (2, 6)


## 6. Stack

Stack introduces a new axis. Decide whether it denotes a component, time level, ensemble member, or another quantity.

In [8]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

S0 = np.stack([a, b], axis=0)
S1 = np.stack([a, b], axis=1)

print("axis=0")
print(S0)
print("shape =", S0.shape)

print("\naxis=1")
print(S1)
print("shape =", S1.shape)

axis=0
[[1 2 3]
 [4 5 6]]
shape = (2, 3)

axis=1
[[1 4]
 [2 5]
 [3 6]]
shape = (3, 2)


$$
\boxed{
\texttt{concatenate}=
\text{existing axis extension}
}
$$

$$
\boxed{
\texttt{stack}=
\text{new axis creation}
}
$$

## 7. Construct a vector field

Stack scalar component arrays on the declared component axis. Keep physical components distinct from spatial indices.

```python
vx.shape == (Nx,Ny)
vy.shape == (Nx,Ny)
```

In [9]:
Nx, Ny = 4, 5

vx = np.ones((Nx, Ny))
vy = 2*np.ones((Nx, Ny))

V = np.stack([vx, vy], axis=-1)

print("V shape =", V.shape)
print("V[0,0] =", V[0,0])

V shape = (4, 5, 2)
V[0,0] = [1. 2.]


$$
(v_x,v_y)
\rightarrow
V_{ij\alpha}
$$

## 8. Column and row stacking

Column_stack forms columns from one-dimensional inputs. Use vstack for vertical stacking; row_stack is deprecated in newer NumPy versions.

In [10]:
e1 = np.array([1., 0.])
e2 = np.array([0., 1.])

Q = np.column_stack([e1, e2])

print(Q)
print("shape =", Q.shape)

[[1. 0.]
 [0. 1.]]
shape = (2, 2)


## 9. Permute axes

Transpose changes axis order. A permutation must carry coordinate and component semantics with it.

```python
A.shape == (2,3,4)
```

```python
np.transpose(A, (2,0,1))
```

```python
(4,2,3)
```

In [11]:
A = np.zeros((2, 3, 4))
B = np.transpose(A, (2, 0, 1))

print("A shape =", A.shape)
print("B shape =", B.shape)

A shape = (2, 3, 4)
B shape = (4, 2, 3)


## 10. Dot-T on higher-dimensional arrays

Dot-T reverses all axes. For a tensor field this generally moves spatial axes as well as transposing components.

In [12]:
A = np.zeros((2, 3, 4))

print("A shape   =", A.shape)
print("A.T shape =", A.T.shape)

A shape   = (2, 3, 4)
A.T shape = (4, 3, 2)


## 11. Swap two axes

Swapaxes exchanges a selected pair. It is useful for transposing tensor components without moving location axes.

```python
T.shape == (Nx,Ny,3,3)
```

In [13]:
T = np.zeros((4, 5, 3, 3))
Tt = np.swapaxes(T, -1, -2)

print("T shape  =", T.shape)
print("Tt shape =", Tt.shape)

T shape  = (4, 5, 3, 3)
Tt shape = (4, 5, 3, 3)


## 12. Move an axis

Moveaxis relocates a component or batch axis while preserving the relative order of the others. Use it at explicitly documented interfaces.

```python
V.shape == (3,Nx,Ny)
```

```python
(Nx,Ny,3)
```

In [14]:
V_first = np.zeros((3, 4, 5))
V_last = np.moveaxis(V_first, 0, -1)

print("component first =", V_first.shape)
print("component last  =", V_last.shape)

component first = (3, 4, 5)
component last  = (4, 5, 3)


## 13. Squeeze

Squeeze removes singleton axes. Specify the intended axis when removing it so a meaningful component or batch axis is not lost accidentally.

```python
(4,5,1)
→
(4,5)
```

In [15]:
A = np.zeros((4, 5, 1))
B = np.squeeze(A)

print(A.shape, "->", B.shape)

(4, 5, 1) -> (4, 5)


## 14. Expand dimensions

Expand_dims inserts a singleton axis. It expresses intended broadcasting rather than generating additional samples.

```python
np.expand_dims(a, axis=-1)
```

In [16]:
a = np.zeros((4, 5))

b = np.expand_dims(a, axis=-1)

print(a.shape, "->", b.shape)

(4, 5) -> (4, 5, 1)


## 15. Views and copies

An array view aliases storage through a different shape or stride pattern. Copy when independent state is required.

```python
np.shares_memory(a,b)
```

In [17]:
A = np.arange(12).reshape(3, 4)

B = A[:, 1:3]
C = A[:, [1, 2]]
D = A.copy()

print("slice shares memory?", np.shares_memory(A, B))
print("fancy shares memory?", np.shares_memory(A, C))
print("copy shares memory? ", np.shares_memory(A, D))

slice shares memory? True
fancy shares memory? False
copy shares memory?  False


## 16. Advanced indexing copies

Advanced integer and boolean indexing returns copied selections. Assigning through an indexed target is a separate operation on the original array.

```python
B = A[:,1:3]
```

```python
C = A[:,[1,2]]
```

## 17. Contiguous storage

Contiguity describes whether entries follow a simple memory order. A transposed view may be valid yet noncontiguous.

In [18]:
A = np.arange(24).reshape(4, 6)
B = A[:, ::2]

print("A C-contiguous?", A.flags["C_CONTIGUOUS"])
print("B C-contiguous?", B.flags["C_CONTIGUOUS"])

A C-contiguous? True
B C-contiguous? False


## 18. Ascontiguousarray

This supplies C-contiguous storage, copying when needed. It can simplify compiled interfaces but does not repair an incorrect physical axis order.

In [19]:
C = np.ascontiguousarray(B)

print("B contiguous?", B.flags["C_CONTIGUOUS"])
print("C contiguous?", C.flags["C_CONTIGUOUS"])

B contiguous? False
C contiguous? True


## 19. Broadcasted vectorization

Express independent arithmetic with aligned arrays. Broadcasting may avoid storing repeated inputs while still creating a large output or temporary.

```python
for i:
    for j:
        U[i,j] = x[i] + 2*y[j]
```

```python
U = x[:,None] + 2*y[None,:]
```

In [20]:
x = np.linspace(0, 1, 4)
y = np.linspace(0, 2, 5)

U = x[:, None] + 2*y[None, :]

print(U)

[[0.         1.         2.         3.         4.        ]
 [0.33333333 1.33333333 2.33333333 3.33333333 4.33333333]
 [0.66666667 1.66666667 2.66666667 3.66666667 4.66666667]
 [1.         2.         3.         4.         5.        ]]


## 20. Einstein summation

Use explicit indices and output labels to state contractions. The array expression should match a written component formula.

$$
y_i=A_{ij}x_j
$$

In [21]:
A = np.array([
    [2., 1.],
    [3., 4.],
])
x = np.array([5., 6.])

y = np.einsum("ij,j->i", A, x)

print(y)

[16. 39.]


$$
C_{ik}=A_{ij}B_{jk}
$$

In [22]:
B = np.array([
    [1., 0.],
    [2., 1.],
])

C = np.einsum("ij,jk->ik", A, B)

print(C)

[[ 4.  1.]
 [11.  4.]]


## 21. Double contraction

Contract both component indices while retaining sample axes. Check a small example against an explicit sum.

$$
A:B=
A_{ij}B_{ij}
$$

In [23]:
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])

c = np.einsum("ij,ij->", A, B)

print(c)

70.0


## 22. Tensor-field contraction

Ellipsis preserves leading location or batch axes. Verify that coefficients and vectors occupy compatible locations and bases.

$$
q_i(\mathbf x)=
-D_{ij}(\mathbf x)g_j(\mathbf x)
$$

In [24]:
Nx, Ny = 4, 5

D = np.zeros((Nx, Ny, 2, 2))
D[...,0,0] = 2.0
D[...,1,1] = 1.0
D[...,0,1] = 0.3
D[...,1,0] = 0.3

g = np.ones((Nx, Ny, 2))

q = -np.einsum("...ij,...j->...i", D, g)

print("D shape =", D.shape)
print("g shape =", g.shape)
print("q shape =", q.shape)

D shape = (4, 5, 2, 2)
g shape = (4, 5, 2)
q shape = (4, 5, 2)


## 23. Tensordot

Choose contracted axes explicitly and inspect output ordering. It is useful for general contractions but can obscure field semantics without documentation.

In [25]:
A = np.arange(6).reshape(2, 3)
x = np.array([1., 2., 3.])

y = np.tensordot(A, x, axes=([1], [0]))

print(y)

[ 8. 26.]


## 24. Batched matrix multiplication

Leading axes represent batches and the last two axes represent matrices. For batched vector solves, an explicit final singleton axis avoids NumPy 2 ambiguity.

In [26]:
A = np.zeros((4, 5, 2, 2))
A[...,0,0] = 2.0
A[...,1,1] = 3.0

x = np.ones((4, 5, 2, 1))

y = A @ x

print("A =", A.shape)
print("x =", x.shape)
print("y =", y.shape)

A = (4, 5, 2, 2)
x = (4, 5, 2, 1)
y = (4, 5, 2, 1)


## 25. Einsum or at

Use at for clear standard matrix products and einsum for explicit component contractions. Neither choice guarantees better performance for every shape.

```python
A @ x
```

```python
np.einsum("...ij,...j->...i", A, x)
```

$$
\boxed{
\text{shortest code}
\neq
\text{always clearest code}
}
$$

## 26. Reductions

Sum, mean, minimum, and maximum remove specified axes. Physical integrals and averages require weights when spacing or volume varies.

In [27]:
A = np.arange(24).reshape(2, 3, 4)

print("A shape =", A.shape)
print("sum axis=0 =", A.sum(axis=0).shape)
print("sum axis=1 =", A.sum(axis=1).shape)
print("sum axis=2 =", A.sum(axis=2).shape)

A shape = (2, 3, 4)
sum axis=0 = (3, 4)
sum axis=1 = (2, 4)
sum axis=2 = (2, 3)


## 27. Retain reduced axes

Keepdims preserves singleton axes for later broadcasting. This makes the relation between a reduced statistic and its original field easier to read.

In [28]:
A = np.arange(24).reshape(2, 3, 4)

m = A.mean(axis=2, keepdims=True)

print("shape =", m.shape)

shape = (2, 3, 1)


## 28. Indices of extrema

Argmin and argmax return locations in an indexing convention. Convert a flattened index back to coordinate indices before interpreting it physically.

In [29]:
u = np.array([0.2, 0.5, 0.1, 0.8])

print("argmin =", np.argmin(u))
print("argmax =", np.argmax(u))

argmin = 2
argmax = 3


## 29. Extract locations

Where with a condition identifies matching indices. Keep these indices paired unless a Cartesian product is intended.

In [30]:
u = np.array([0.2, 0.5, 0.1, 0.8])

idx = np.where(u > 0.4)[0]

print(idx)

[1 3]


## 30. Broadcast_to

Broadcast_to creates a read-only view with repeated conceptual values. Do not treat it as an independent writable copy.

In [31]:
a = np.array([1., 2., 3.])

B = np.broadcast_to(a, (4, 3))

print(B)
print("shape =", B.shape)

[[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]
shape = (4, 3)


## 31. Tile versus broadcasting

Tile physically repeats entries; broadcasting usually avoids that repeated input storage. The resulting arithmetic can still allocate a large array.

In [32]:
a = np.array([1., 2., 3.])

T = np.tile(a, (1000, 1))
B = np.broadcast_to(a, (1000, 3))

print("tile nbytes      =", T.nbytes)
print("broadcast nbytes =", B.nbytes)
print("broadcast shares memory with a?", np.shares_memory(a, B))

tile nbytes      = 24000
broadcast nbytes = 24000
broadcast shares memory with a? True


## 32. Limit temporary arrays

Estimate peak memory for intermediate results, not only input sizes. Chunking or an explicit loop can be the clearer efficient choice.

```python
q = -D * gradU
```

$$
\boxed{
\text{correctness}
\rightarrow
\text{clarity}
\rightarrow
\text{performance}
}
$$

## 33. In-place operations

In-place arithmetic saves some allocation but modifies shared storage and must respect dtype casting. Verify that old values are not needed later.

```python
U *= 2
```

In [33]:
U = np.array([1., 2., 3.])
V = U.copy()

U *= 2

print("U =", U)
print("V =", V)

U = [2. 4. 6.]
V = [1. 2. 3.]


## 34. Dtype and memory

Choose precision by accuracy needs and then measure memory and performance. Lower precision is not automatically adequate for ill-conditioned problems.

In [34]:
A64 = np.zeros((1000,1000), dtype=np.float64)
A32 = np.zeros((1000,1000), dtype=np.float32)

print("float64 MB =", A64.nbytes/1024**2)
print("float32 MB =", A32.nbytes/1024**2)

float64 MB = 7.62939453125
float32 MB = 3.814697265625


## 35. Translate a loop by its dependency structure

Identify loop indices, read and write locations, reductions, and sequential dependencies. Only independent updates can be replaced by simultaneous array arithmetic without changing the method.

$$
\boxed{
\text{iteration index}
\rightarrow
\text{array axis}
}
$$

$$
\boxed{
\text{repeated scalar operation}
\rightarrow
\text{ufunc / broadcasting / reduction / contraction}
}
$$

## 35.1. Read the dependency pattern first

Identify indices, data locations, write dependencies, and reductions before choosing an array expression.

$$
\boxed{
y_i=f(x_i)
\quad\Longrightarrow\quad
\mathbf y=f(\mathbf x)
}
$$

$$
x_{n+1}=F(x_n)
$$

## 36. Pattern A: elementwise laws

Ufuncs evaluate independent scalar formulas across arrays. Polynomial laws, diffusivity fields, and coordinate transformations illustrate this pattern.

### Example 1: Polynomial evaluation

In [35]:
x = np.linspace(-2, 2, 8)

y_loop = np.empty_like(x)
for i in range(x.size):
    y_loop[i] = x[i]**2 + 2*x[i] + 1

y_vec = x**2 + 2*x + 1

print("loop =", y_loop)
print("vec  =", y_vec)
print("same =", scaled_allclose(y_loop, y_vec))

loop = [1.         0.18367347 0.02040816 0.51020408 1.65306122 3.44897959
 5.89795918 9.        ]
vec  = [1.         0.18367347 0.02040816 0.51020408 1.65306122 3.44897959
 5.89795918 9.        ]
same = True


### Example 2: Combined ufuncs

In [36]:
x = np.linspace(0, 2, 8)

y_loop = np.empty_like(x)
for i in range(x.size):
    y_loop[i] = np.exp(-x[i]) * np.sin(2*x[i])

y_vec = np.exp(-x) * np.sin(2*x)

print("same =", scaled_allclose(y_loop, y_vec))

same = True


### Example 3: An elementwise relation between arrays

In [37]:
a = np.arange(1., 7.)
b = np.linspace(10., 20., 6)

y_loop = np.empty_like(a)
for i in range(a.size):
    y_loop[i] = 3*a[i] - 0.5*b[i]

y_vec = 3*a - 0.5*b

print("same =", scaled_allclose(y_loop, y_vec))

same = True


### Example 4: A nonlinear diffusivity field

In [38]:
M = np.linspace(0.08, 0.35, 10)
D0 = 1e-10
a = 8.0
Mref = 0.15

D_loop = np.empty_like(M)
for i in range(M.size):
    D_loop[i] = D0*np.exp(a*(M[i]-Mref))

D_vec = D0*np.exp(a*(M-Mref))

print("D =", D_vec)
print("same =", scaled_allclose(D_loop, D_vec))

D = [5.71209064e-11 7.26149037e-11 9.23116346e-11 1.17351087e-10
 1.49182470e-10 1.89648088e-10 2.41089971e-10 3.06485420e-10
 3.89619330e-10 4.95303242e-10]
same = True


### Example 5: Polar-to-Cartesian coordinates

In [39]:
theta = np.linspace(0, 2*np.pi, 9)
r = 2.0 + 0.2*np.cos(2*theta)

x_loop = np.empty_like(theta)
y_loop = np.empty_like(theta)
for i in range(theta.size):
    x_loop[i] = r[i]*np.cos(theta[i])
    y_loop[i] = r[i]*np.sin(theta[i])

x_vec = r*np.cos(theta)
y_vec = r*np.sin(theta)

print("x same =", scaled_allclose(x_loop, x_vec))
print("y same =", scaled_allclose(y_loop, y_vec))

x same = True
y same = True


## 37. Pattern B: conditions and masks

Clip, where, boolean assignment, and select express different piecewise operations. Where and select evaluate their branch arrays eagerly; guard invalid arithmetic explicitly.

### Example 6: Bounds with clip

In [40]:
x = np.array([-2., -0.5, 0.2, 0.8, 1.5, 3.0])

y_loop = np.empty_like(x)
for i in range(x.size):
    y_loop[i] = min(max(x[i], 0.0), 1.0)

y_vec = np.clip(x, 0.0, 1.0)

print("loop =", y_loop)
print("vec  =", y_vec)

loop = [0.  0.  0.2 0.8 1.  1. ]
vec  = [0.  0.  0.2 0.8 1.  1. ]


### Example 7: A two-regime law with where

In [41]:
M = np.linspace(0.10, 0.50, 9)
Mcrit = 0.30

beta_loop = np.empty_like(M)
for i in range(M.size):
    if M[i] <= Mcrit:
        beta_loop[i] = 0.20
    else:
        beta_loop[i] = 0.05

beta_vec = np.where(M <= Mcrit, 0.20, 0.05)

print("same =", scaled_allclose(beta_loop, beta_vec))

same = True


### Example 8: Masked assignment

In [42]:
x = np.array([-3., -1., 0., 2., 4.])

y_loop = x.copy()
for i in range(x.size):
    if y_loop[i] > 0:
        y_loop[i] = y_loop[i]**2

y_vec = x.copy()
mask = y_vec > 0
y_vec[mask] = y_vec[mask]**2

print("loop =", y_loop)
print("vec  =", y_vec)

loop = [-3. -1.  0.  4. 16.]
vec  = [-3. -1.  0.  4. 16.]


### Example 9: A three-regime law with select

In [43]:
M = np.linspace(0.05, 0.55, 11)

k_loop = np.empty_like(M)
for i in range(M.size):
    if M[i] < 0.20:
        k_loop[i] = 1.0
    elif M[i] < 0.40:
        k_loop[i] = 2.0
    else:
        k_loop[i] = 5.0

conditions = [M < 0.20, M < 0.40]
choices = [1.0, 2.0]
k_vec = np.select(conditions, choices, default=5.0)

print("same =", scaled_allclose(k_loop, k_vec))

same = True


### Example 10: NaN handling

In [44]:
x = np.array([1.0, np.nan, 3.0, np.inf, -np.inf])

y_loop = x.copy()
for i in range(x.size):
    if np.isnan(y_loop[i]):
        y_loop[i] = 0.0
    elif np.isposinf(y_loop[i]):
        y_loop[i] = 99.0
    elif np.isneginf(y_loop[i]):
        y_loop[i] = -99.0

y_vec = np.nan_to_num(x, nan=0.0, posinf=99.0, neginf=-99.0)

print("loop =", y_loop)
print("vec  =", y_vec)

loop = [  1.   0.   3.  99. -99.]
vec  = [  1.   0.   3.  99. -99.]


## 38. Pattern C: reductions

Replace independent accumulations with reductions on the intended axis. For vector normalization, handle zero norms rather than dividing blindly.

$$
\boxed{
\sum_i,\;\max_i,\;\mathrm{mean}_i
\quad\Longrightarrow\quad
\texttt{axis=...}
}
$$

### Example 11: Row sums

In [45]:
A = np.arange(1., 13.).reshape(4, 3)

s_loop = np.empty(A.shape[0])
for i in range(A.shape[0]):
    s_loop[i] = 0.0
    for j in range(A.shape[1]):
        s_loop[i] += A[i, j]

s_vec = A.sum(axis=1)

print("same =", scaled_allclose(s_loop, s_vec))

same = True


### Example 12: Column means

In [46]:
A = np.arange(1., 13.).reshape(4, 3)

m_loop = np.empty(A.shape[1])
for j in range(A.shape[1]):
    total = 0.0
    for i in range(A.shape[0]):
        total += A[i, j]
    m_loop[j] = total/A.shape[0]

m_vec = A.mean(axis=0)

print("same =", scaled_allclose(m_loop, m_vec))

same = True


### Example 13: Vector norms

In [47]:
V = np.array([[3.,4.,0.],
              [1.,2.,2.],
              [2.,0.,0.]])

n_loop = np.empty(V.shape[0])
for i in range(V.shape[0]):
    n_loop[i] = np.sqrt(np.sum(V[i]**2))

n_vec = np.linalg.norm(V, axis=1)

print("loop =", n_loop)
print("vec  =", n_vec)

loop = [5. 3. 2.]
vec  = [5. 3. 2.]


### Example 14: Normalize multiple vectors

In [48]:
V = np.array([[3.,4.,0.],
              [1.,2.,2.],
              [2.,0.,0.]])

Vn_loop = np.empty_like(V)
for i in range(V.shape[0]):
    Vn_loop[i] = V[i]/np.linalg.norm(V[i])

norms = np.linalg.norm(V, axis=1, keepdims=True)
Vn_vec = V/norms

print("same =", scaled_allclose(Vn_loop, Vn_vec))
print("row norms =", np.linalg.norm(Vn_vec, axis=1))

same = True
row norms = [1. 1. 1.]


### Example 15: Weighted sums

In [49]:
A = np.arange(1., 13.).reshape(4, 3)
w = np.array([0.2, 0.3, 0.5])

y_loop = np.empty(A.shape[0])
for i in range(A.shape[0]):
    y_loop[i] = 0.0
    for j in range(A.shape[1]):
        y_loop[i] += A[i, j]*w[j]

y_vec = A @ w

print("same =", scaled_allclose(y_loop, y_vec))

same = True


## 39. Pattern D: differences and stencils

Diff and aligned slices express neighbour relationships. State spacing, axis order, boundary handling, and whether values are nodes, cells, or faces.

$$
u_{i+1}-u_i
$$

```python
u[1:] - u[:-1]
```

### Example 16: Forward differences with diff

In [50]:
u = np.array([0., 1., 4., 9., 16.])

du_loop = np.empty(u.size-1)
for i in range(u.size-1):
    du_loop[i] = u[i+1] - u[i]

du_vec = np.diff(u)

print("loop =", du_loop)
print("vec  =", du_vec)

loop = [1. 3. 5. 7.]
vec  = [1. 3. 5. 7.]


### Example 17: Centred first differences

In [51]:
x = np.linspace(0, 1, 11)
u = x**3
dx = x[1]-x[0]

dudx_loop = np.full_like(u, np.nan)
for i in range(1, u.size-1):
    dudx_loop[i] = (u[i+1]-u[i-1])/(2*dx)

dudx_vec = np.full_like(u, np.nan)
dudx_vec[1:-1] = (u[2:]-u[:-2])/(2*dx)

print("same interior =", scaled_allclose(dudx_loop[1:-1], dudx_vec[1:-1]))

same interior = True


### Example 18: A one-dimensional Laplacian

In [52]:
x = np.linspace(0, 1, 11)
u = np.sin(np.pi*x)
dx = x[1]-x[0]

lap_loop = np.full_like(u, np.nan)
for i in range(1, u.size-1):
    lap_loop[i] = (u[i+1]-2*u[i]+u[i-1])/dx**2

lap_vec = np.full_like(u, np.nan)
lap_vec[1:-1] = (u[2:] - 2*u[1:-1] + u[:-2])/dx**2

print("same interior =", scaled_allclose(lap_loop[1:-1], lap_vec[1:-1]))

same interior = True


### Example 19: A two-dimensional Laplacian

In [53]:
ny, nx = 6, 8
U = np.arange(ny*nx, dtype=float).reshape(ny, nx)
dx = dy = 1.0

lap_loop = np.full_like(U, np.nan)
for j in range(1, ny-1):
    for i in range(1, nx-1):
        lap_loop[j, i] = (
            (U[j, i+1]-2*U[j, i]+U[j, i-1])/dx**2
            + (U[j+1, i]-2*U[j, i]+U[j-1, i])/dy**2
        )

lap_vec = np.full_like(U, np.nan)
lap_vec[1:-1,1:-1] = (
    (U[1:-1,2:] - 2*U[1:-1,1:-1] + U[1:-1,:-2])/dx**2
    + (U[2:,1:-1] - 2*U[1:-1,1:-1] + U[:-2,1:-1])/dy**2
)

print("same interior =", scaled_allclose(
    lap_loop[1:-1,1:-1],
    lap_vec[1:-1,1:-1]
))

same interior = True


### Example 20: Flux divergence

In [54]:
ny, nx = 6, 8
qx = np.random.default_rng(0).normal(size=(ny, nx))
qy = np.random.default_rng(1).normal(size=(ny, nx))
dx = dy = 1.0

div_loop = np.full_like(qx, np.nan)
for j in range(1, ny-1):
    for i in range(1, nx-1):
        div_loop[j, i] = (
            (qx[j, i+1]-qx[j, i-1])/(2*dx)
            + (qy[j+1, i]-qy[j-1, i])/(2*dy)
        )

div_vec = np.full_like(qx, np.nan)
div_vec[1:-1,1:-1] = (
    (qx[1:-1,2:] - qx[1:-1,:-2])/(2*dx)
    + (qy[2:,1:-1] - qy[:-2,1:-1])/(2*dy)
)

print("same interior =", scaled_allclose(
    div_loop[1:-1,1:-1],
    div_vec[1:-1,1:-1]
))

same interior = True


## 40. Pattern E: pairwise geometry

Broadcasting constructs outer products, pairwise distances, and radial kernels. Pairwise output memory grows with the product of the two set sizes.

$$
f(x_i,y_j)
\quad\Longrightarrow\quad
f(x[:,None],y[None,:])
$$

### Example 21: Outer products

In [55]:
a = np.array([1.,2.,3.])
b = np.array([10.,20.,30.,40.])

O_loop = np.empty((a.size, b.size))
for i in range(a.size):
    for j in range(b.size):
        O_loop[i,j] = a[i]*b[j]

O_vec = a[:,None]*b[None,:]

print("same =", scaled_allclose(O_loop, O_vec))

same = True


### Example 22: Pairwise point distances

In [56]:
P = np.array([[0.,0.],
              [1.,0.],
              [0.,1.]])
Q = np.array([[1.,1.],
              [2.,0.]])

D_loop = np.empty((P.shape[0], Q.shape[0]))
for i in range(P.shape[0]):
    for j in range(Q.shape[0]):
        D_loop[i,j] = np.linalg.norm(P[i]-Q[j])

diff = P[:,None,:] - Q[None,:,:]
D_vec = np.linalg.norm(diff, axis=2)

print("same =", scaled_allclose(D_loop, D_vec))
print(D_vec)

same = True
[[1.41421356 2.        ]
 [1.         1.        ]
 [1.         2.23606798]]


### Example 23: A radial distance field

In [57]:
x = np.linspace(-1,1,5)
y = np.linspace(-2,2,7)
xc, yc = 0.2, -0.3

R_loop = np.empty((y.size, x.size))
for j in range(y.size):
    for i in range(x.size):
        R_loop[j,i] = np.sqrt((x[i]-xc)**2 + (y[j]-yc)**2)

R_vec = np.sqrt(
    (x[None,:]-xc)**2
    + (y[:,None]-yc)**2
)

print("same =", scaled_allclose(R_loop, R_vec))
print("shape =", R_vec.shape)

same = True
shape = (7, 5)


### Example 24: Pairwise radial kernels

In [58]:
x = np.linspace(0, 1, 6)
L = 0.2

K_loop = np.empty((x.size, x.size))
for i in range(x.size):
    for j in range(x.size):
        r = abs(x[i]-x[j])
        K_loop[i,j] = np.exp(-r/L)

R = np.abs(x[:,None] - x[None,:])
K_vec = np.exp(-R/L)

print("same =", scaled_allclose(K_loop, K_vec))

same = True


## 41. Pattern F: batched linear algebra

Batch rotations, tensor actions, stresses, energies, solves, and Fourier sums. A fourth-order stiffness contraction must use tensor components or a documented engineering-shear convention.

### Example 25: Rotate multiple vectors

In [59]:
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

X = np.array([[1.,0.],
              [0.,1.],
              [1.,1.],
              [2.,-1.]])

Y_loop = np.empty_like(X)
for p in range(X.shape[0]):
    Y_loop[p] = R @ X[p]

Y_vec = X @ R.T

print("same =", scaled_allclose(Y_loop, Y_vec))

same = True


### Example 26: Batched matrix-vector products

In [60]:
A = np.array([
    [[2.,0.],[0.,1.]],
    [[1.,1.],[0.,2.]],
    [[3.,0.5],[0.5,1.]]
])
x = np.array([[1.,2.],
              [3.,4.],
              [5.,6.]])

y_loop = np.empty_like(x)
for p in range(A.shape[0]):
    y_loop[p] = A[p] @ x[p]

y_vec = (A @ x[...,None])[...,0]

print("same =", scaled_allclose(y_loop, y_vec))

same = True


### Example 27: Traction at multiple points

In [61]:
sigma = np.array([
    [[10.,2.],[2.,5.]],
    [[8.,1.],[1.,4.]],
    [[6.,0.],[0.,3.]]
])
n = np.array([
    [1.,0.],
    [0.,1.],
    [1/np.sqrt(2), 1/np.sqrt(2)]
])

t_loop = np.empty_like(n)
for p in range(sigma.shape[0]):
    t_loop[p] = sigma[p] @ n[p]

t_vec = np.einsum("pij,pj->pi", sigma, n)

print("same =", scaled_allclose(t_loop, t_vec))
print("traction =\n", t_vec)

same = True
traction =
 [[10.          2.        ]
 [ 1.          4.        ]
 [ 4.24264069  2.12132034]]


### Example 28: Normal and shear traction

In [62]:
sigma = np.array([
    [[10.,2.],[2.,5.]],
    [[8.,1.],[1.,4.]],
    [[6.,0.],[0.,3.]]
])
n = np.array([
    [1.,0.],
    [0.,1.],
    [1/np.sqrt(2), 1/np.sqrt(2)]
])
m = np.column_stack((-n[:,1], n[:,0]))

t = np.einsum("pij,pj->pi", sigma, n)

sn_loop = np.empty(n.shape[0])
tn_loop = np.empty(n.shape[0])
for p in range(n.shape[0]):
    sn_loop[p] = n[p] @ t[p]
    tn_loop[p] = m[p] @ t[p]

sn_vec = np.einsum("pi,pi->p", n, t)
tn_vec = np.einsum("pi,pi->p", m, t)

print("normal same =", scaled_allclose(sn_loop, sn_vec))
print("shear  same =", scaled_allclose(tn_loop, tn_vec))

normal same = True
shear  same = True


### Example 29: A constitutive contraction at multiple points

In [63]:
# 2D Voigt-like toy example: [exx, eyy, gxy]
C = np.array([
    [10.,3.,0.],
    [3.,8.,0.],
    [0.,0.,2.]
])

eps = np.array([
    [0.01, 0.00, 0.002],
    [0.00, 0.02, 0.001],
    [0.01, 0.01, 0.000]
])

sig_loop = np.empty_like(eps)
for p in range(eps.shape[0]):
    sig_loop[p] = C @ eps[p]

sig_vec = eps @ C.T

print("same =", scaled_allclose(sig_loop, sig_vec))
print(sig_vec)

same = True
[[0.1   0.03  0.004]
 [0.06  0.16  0.002]
 [0.13  0.11  0.   ]]


### Example 30: Double contraction and energy

In [64]:
eps = np.array([
    [[0.01,0.002],[0.002,0.005]],
    [[0.02,0.001],[0.001,0.010]]
])
sig = np.array([
    [[100.,20.],[20.,50.]],
    [[80.,10.],[10.,40.]]
])

W_loop = np.empty(eps.shape[0])
for p in range(eps.shape[0]):
    W_loop[p] = 0.5*np.sum(eps[p]*sig[p])

W_vec = 0.5*np.einsum("pij,pij->p", eps, sig)

print("same =", scaled_allclose(W_loop, W_vec))
print("W =", W_vec)

same = True
W = [0.665 1.01 ]


### Example 31: Batched small linear systems

In [65]:
A = np.array([
    [[3.,1.],[1.,2.]],
    [[2.,0.],[0.,4.]],
    [[5.,2.],[2.,3.]]
])
b = np.array([
    [1.,0.],
    [2.,8.],
    [1.,1.]
])

x_loop = np.empty_like(b)
for p in range(A.shape[0]):
    x_loop[p] = np.linalg.solve(A[p], b[p])

x_vec = np.linalg.solve(A, b[...,None])[...,0]

print("same =", scaled_allclose(x_loop, x_vec))
print(x_vec)

same = True
[[ 0.4        -0.2       ]
 [ 1.          2.        ]
 [ 0.09090909  0.27272727]]


### Example 32: Fourier angles and modes

In [66]:
theta = np.linspace(0, 2*np.pi, 200, endpoint=False)
a0 = 10.0
a = np.array([1.0, 0.4, 0.2])
b = np.array([0.3, 0.1, 0.05])

r_loop = np.empty_like(theta)
for j, th in enumerate(theta):
    rj = a0
    for mode in range(1, len(a)+1):
        rj += a[mode-1]*np.cos(mode*th)
        rj += b[mode-1]*np.sin(mode*th)
    r_loop[j] = rj

nmode = np.arange(1, len(a)+1)[:,None]
Theta = theta[None,:]
r_vec = (
    a0
    + (a[:,None]*np.cos(nmode*Theta)).sum(axis=0)
    + (b[:,None]*np.sin(nmode*Theta)).sum(axis=0)
)

print("same =", scaled_allclose(r_loop, r_vec))
print("shape =", r_vec.shape)

same = True
shape = (200,)


## 42. Pattern G: wood and multiphysics illustrations

These algebraic examples show moisture dependence, directional flux, Robin transfer, and mechanism blending. Their parameters are illustrative, and the FSP-related transition is not an experimentally validated constitutive law.

### Example 33: Moisture-dependent anisotropic diffusivity

In [67]:
M = np.linspace(0.08, 0.35, 12).reshape(3,4)
D0 = 1e-10
a = 8.0
Mref = 0.15

Dx_loop = np.empty_like(M)
Dy_loop = np.empty_like(M)
for j in range(M.shape[0]):
    for i in range(M.shape[1]):
        base = D0*np.exp(a*(M[j,i]-Mref))
        Dx_loop[j,i] = base
        Dy_loop[j,i] = 0.8*base

base = D0*np.exp(a*(M-Mref))
Dx_vec = base
Dy_vec = 0.8*base

print("Dx same =", scaled_allclose(Dx_loop, Dx_vec))
print("Dy same =", scaled_allclose(Dy_loop, Dy_vec))

Dx same = True
Dy same = True


### Example 34: Directional flux components

In [68]:
rng = np.random.default_rng(3)
dMdx = rng.normal(size=(4,5))
dMdy = rng.normal(size=(4,5))
Dx = 1e-10*np.ones((4,5))
Dy = 0.8e-10*np.ones((4,5))

qx_loop = np.empty_like(dMdx)
qy_loop = np.empty_like(dMdy)
for j in range(4):
    for i in range(5):
        qx_loop[j,i] = -Dx[j,i]*dMdx[j,i]
        qy_loop[j,i] = -Dy[j,i]*dMdy[j,i]

qx_vec = -Dx*dMdx
qy_vec = -Dy*dMdy

print("qx same =", scaled_allclose(qx_loop, qx_vec))
print("qy same =", scaled_allclose(qy_loop, qy_vec))

qx same = True
qy same = True


### Example 35: Robin boundary transfer

In [69]:
M_surface = np.array([0.25, 0.22, 0.18, 0.15, 0.12])
M_inf = 0.10
h = 1e-7

q_loop = np.empty_like(M_surface)
for i in range(M_surface.size):
    q_loop[i] = h*(M_surface[i]-M_inf)

q_vec = h*(M_surface-M_inf)

print("same =", scaled_allclose(q_loop, q_vec))

same = True


### Example 36: An illustrative connectivity transition near FSP

In [70]:
M = np.linspace(0.15, 0.50, 12)
Mcw = 0.30
Mc = 0.40

chi_loop = np.empty_like(M)
for i in range(M.size):
    if M[i] <= Mcw:
        chi_loop[i] = 0.0
    elif M[i] >= Mc:
        chi_loop[i] = 1.0
    else:
        xi = (M[i]-Mcw)/(Mc-Mcw)
        chi_loop[i] = 3*xi**2 - 2*xi**3

xi = np.clip((M-Mcw)/(Mc-Mcw), 0.0, 1.0)
chi_vec = 3*xi**2 - 2*xi**3

print("same =", scaled_allclose(chi_loop, chi_vec))
print("chi =", chi_vec)

same = True
chi = [0.         0.         0.         0.         0.         0.02329076
 0.36513899 0.8174305  1.         1.         1.         1.        ]


### Example 37: Relative and effective permeability

In [71]:
S = np.linspace(0, 1, 11)
k_intrinsic = 2e-14
n = 3.0

keff_loop = np.empty_like(S)
for i in range(S.size):
    krl = S[i]**n
    keff_loop[i] = k_intrinsic*krl

krl_vec = S**n
keff_vec = k_intrinsic*krl_vec

print("same =", scaled_allclose(keff_loop, keff_vec))

same = True


### Example 38: Illustrative mechanism blending

In [72]:
M = np.linspace(0.15, 0.55, 20)

bulk_loop = np.empty_like(M)
film_loop = np.empty_like(M)
vapor_loop = np.empty_like(M)

for i in range(M.size):
    bulk_loop[i] = 1/(1 + np.exp(-(M[i]-0.40)/0.02))
    film_loop[i] = np.exp(-((M[i]-0.35)/0.05)**2)
    vapor_loop[i] = 1/(1 + np.exp((M[i]-0.35)/0.03))

bulk_vec = 1/(1 + np.exp(-(M-0.40)/0.02))
film_vec = np.exp(-((M-0.35)/0.05)**2)
vapor_vec = 1/(1 + np.exp((M-0.35)/0.03))

print("bulk same =", scaled_allclose(bulk_loop, bulk_vec))
print("film same =", scaled_allclose(film_loop, film_vec))
print("vapor same =", scaled_allclose(vapor_loop, vapor_vec))

bulk same = True
film same = True
vapor same = True


## 43. Pattern H: groups and scatter

Bincount aggregates groups; add.at accumulates repeated indices; sliding windows expose local samples. Ordinary indexed addition can lose repeated-index contributions.

### Example 39: Group sums with bincount

In [73]:
group = np.array([0,0,1,1,1,2,2,3])
value = np.array([1.,2.,3.,4.,5.,6.,7.,8.])

sum_loop = np.zeros(4)
for i in range(value.size):
    sum_loop[group[i]] += value[i]

sum_vec = np.bincount(group, weights=value, minlength=4)

print("loop =", sum_loop)
print("vec  =", sum_vec)

loop = [ 3. 12. 13.  8.]
vec  = [ 3. 12. 13.  8.]


### Example 40: Repeated-index scatter with add.at

In [74]:
node = np.array([0,1,1,2,2,3])
contrib = np.array([1.,2.,3.,4.,5.,6.])

a_loop = np.zeros(4)
for e in range(node.size):
    a_loop[node[e]] += contrib[e]

a_vec = np.zeros(4)
np.add.at(a_vec, node, contrib)

print("loop =", a_loop)
print("vec  =", a_vec)

loop = [1. 5. 9. 6.]
vec  = [1. 5. 9. 6.]


### Example 41: Moving averages

In [75]:
x = np.arange(10., dtype=float)
w = 3

ma_loop = np.empty(x.size-w+1)
for i in range(x.size-w+1):
    ma_loop[i] = x[i:i+w].mean()

windows = np.lib.stride_tricks.sliding_window_view(x, w)
ma_vec = windows.mean(axis=1)

print("same =", scaled_allclose(ma_loop, ma_vec))
print("moving mean =", ma_vec)

same = True
moving mean = [1. 2. 3. 4. 5. 6. 7. 8.]


### Example 42: Histogram counts

In [76]:
x = np.array([0.1,0.2,0.2,0.4,0.7,0.8,0.9])
edges = np.array([0.0,0.25,0.5,0.75,1.0])

count_loop = np.zeros(len(edges)-1, dtype=int)
for value in x:
    for j in range(len(edges)-1):
        if edges[j] <= value < edges[j+1]:
            count_loop[j] += 1
            break

count_vec, _ = np.histogram(x, bins=edges)

print("loop =", count_loop)
print("vec  =", count_vec)

loop = [3 1 1 2]
vec  = [3 1 1 2]


## 44. Memory limits and hybrid strategies

Use chunk loops for large pairwise calculations and control loops for time steps or Newton iterations. Vectorize independent spatial work within those loops.

## 44.1. Large temporary arrays

A compact pairwise expression can allocate a prohibitively large intermediate. Estimate the output and temporary sizes first.

```python
diff = X[:,None,:] - X[None,:,:]
```

### Example 43: Chunked pairwise calculations

$$
\boxed{
\text{algorithmic loop}
+
\text{array vectorization}
}
$$

### Example 44: Time control with vectorized spatial work

```python
for n in range(Nt):
    lap[1:-1] = (
        u[2:] - 2*u[1:-1] + u[:-2]
    ) / dx**2

    u_new[1:-1] = u[1:-1] + dt*D*lap[1:-1]
    u, u_new = u_new, u
```

$$
u^{n+1}=F(u^n)
$$

### Example 45: Retain Newton iteration as a control loop

## 44.2. A vectorization decision checklist

Check equivalence, dependency order, memory, readability, and representative timing before accepting a replacement.

| Loop structure | NumPy pattern to consider |  
| --- | --- |  
| Apply the same function independently | ufunc / elementwise |  
| Assign values by condition | mask / `where` / `select` |  
| Sum, mean, or maximum | reduction + `axis` |  
| Rowwise norm | `np.linalg.norm(..., axis=...)` |  
| All pairwise combinations | broadcasting |  
| Repeated $A_i x_i$ | batch `@` / `einsum` |  
| tensor contraction | `einsum` / `tensordot` |  
| finite difference | slicing / `diff` |  
| moving stencil | slicing / `sliding_window_view` |  
| group sum | `bincount` |  
| repeated-index accumulation | `add.at` |  
| recursive/time iteration | Retain the loop |  
| Very large pairwise calculation | chunked vectorization |

## 45. Timing

Timing is machine, layout, and problem-size dependent. Confirm equivalent results first, repeat measurements, and interpret a single benchmark cautiously.

In [77]:
from time import perf_counter

N = 200_000
x = np.linspace(0, 1, N)

t0 = perf_counter()
y_loop = np.empty_like(x)
for i in range(N):
    y_loop[i] = np.sin(x[i]) + x[i]**2
t_loop = perf_counter() - t0

t0 = perf_counter()
y_vec = np.sin(x) + x**2
t_vec = perf_counter() - t0

print("same?", scaled_allclose(y_loop, y_vec))
print("loop time =", t_loop)
print("vectorized time =", t_vec)

same? True
loop time = 0.15112103699993895
vectorized time = 0.0030759379997107317


## 46. Why some loops remain

Time evolution and nonlinear iteration are often inherently sequential at this algorithmic level. Eliminating every loop is not a scientific objective.

$$
\boxed{
\text{algorithmic loop}
\neq
\text{array arithmetic loop}
}
$$

## 47. Read advanced array code

Track shape, axis meaning, aliasing, component contractions, and temporary memory. These checks are more informative than the expression's length.

## 48. Advanced-operation map

Reshape changes indexing, stack adds an axis, moveaxis reorders axes, and contraction removes component indices. Keep these transformations conceptually distinct.

| Purpose | NumPy |  
| --- | --- |  
| Change shape | `reshape` |  
| Flatten to one dimension | `ravel`, `flatten` |  
| Join an existing axis | `concatenate` |  
| Create a new axis | `stack` |  
| column basis matrix | `column_stack` |  
| axis permutation | `transpose` |  
| Swap two axes | `swapaxes` |  
| Move an axis | `moveaxis` |  
| Remove a singleton axis | `squeeze` |  
| Add a singleton axis | `expand_dims`, `None` |  
| matrix/tensor contraction | `@`, `tensordot`, `einsum` |  
| reduction | `sum`, `mean`, `max`, `min` |  
| Check memory sharing | `shares_memory` |  
| contiguous copy | `ascontiguousarray` |  
| broadcasted view | `broadcast_to` |

| Purpose | NumPy |  
| --- | --- |  
| Vectorize elementwise work | ufunc, broadcasting |  
| Vectorize conditional assignment | boolean mask, `where`, `select`, `clip` |  
| Reduce along an axis | `sum`, `mean`, `max`, `min`, `axis` |  
| stencil / finite difference | slicing, `diff` |  
| batch matrix/vector | `@`, `einsum` |  
| Pairwise calculation | broadcasting |  
| group aggregation | `bincount`, `add.at` |  
| moving window | `sliding_window_view` |

## 49. Ideas to retain

Preserve dependency order, field location, and component conventions; then measure correctness, memory, and speed. Vectorization is an implementation choice, not a physical model.

## 50. Continue to P7

The next note uses dense linear algebra to connect factorization, residuals, conditioning, eigenproblems, and least squares.

$$
\boxed{
\text{array operation}
\rightarrow
\text{numerical linear algebra algorithm}
}
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [NumPy indexing](https://numpy.org/doc/stable/user/basics.indexing.html)
- [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy linear algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.